# SDH exp_010 — 챔피언 FE 단계별 ablation

exp009 최고 FE에서 챔피언 보고서의 개선 요소를 하나씩 누적합니다. 모델 파라미터는 LR `C=0.07`, `max_iter=2000`으로 고정합니다.


In [ ]:
from pathlib import Path
from time import perf_counter
import json
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'common').is_dir() and (p / 'experiments').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('프로젝트 루트를 찾지 못했습니다.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments.SDH.exp_010_champion_fe_ablation.preprocessing import make_candidates

TRAIN_PATH = PROJECT_ROOT / 'data' / 'raw' / 'train.csv'
RESULTS_DIR = PROJECT_ROOT / 'experiments' / 'SDH' / 'exp_010_champion_fe_ablation' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PARAMS = {
    'solver': 'lbfgs',
    'C': 0.07,
    'max_iter': 2000,
    'class_weight': 'balanced',
    'random_state': 42,
}
N_SPLITS = 5
SCREEN_SEEDS = (42,)
CONFIRMATION_SEEDS = (42, 52, 62)
MODEL_PARAMS

In [ ]:
train = pd.read_csv(TRAIN_PATH, low_memory=False)
X = train.drop(columns=['ID', 'SUBCLASS'])
y = train['SUBCLASS'].reset_index(drop=True)

print(f'train shape: {train.shape}')
print(f'classes: {y.nunique()}')
print(f'minimum class size: {y.value_counts().min()}')
display(train[['ID', 'SUBCLASS']].head())

In [ ]:
candidates = make_candidates()
pd.DataFrame(
    [
        {'case': name, **transformer.get_params(deep=False)}
        for name, transformer in candidates.items()
    ]
).set_index('case')

## 평가 함수

아래 셀에 실제 CV 과정이 공개되어 있습니다. 전처리는 각 fold의 train에서만 `fit`됩니다.

In [ ]:
def evaluate_case(case_name, preprocessor, seeds=SCREEN_SEEDS):
    per_fold = []
    per_seed = []
    convergence_warning_count = 0

    for seed in seeds:
        splitter = StratifiedKFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=seed,
        )
        oof_prediction = np.empty(len(X), dtype=object)

        for fold, (fit_index, valid_index) in enumerate(splitter.split(X, y), start=1):
            started = perf_counter()
            pipeline = Pipeline([
                ('features', clone(preprocessor)),
                ('model', LogisticRegression(**MODEL_PARAMS)),
            ])
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always', ConvergenceWarning)
                pipeline.fit(X.iloc[fit_index], y.iloc[fit_index])
            fold_convergence_warnings = sum(
                issubclass(item.category, ConvergenceWarning) for item in caught
            )
            convergence_warning_count += fold_convergence_warnings

            prediction = pipeline.predict(X.iloc[valid_index])
            oof_prediction[valid_index] = prediction
            feature_count = int(pipeline.named_steps['model'].n_features_in_)
            row = {
                'case': case_name,
                'seed': seed,
                'fold': fold,
                'feature_count': feature_count,
                'f1_macro': f1_score(
                    y.iloc[valid_index], prediction,
                    average='macro', zero_division=0,
                ),
                'accuracy': accuracy_score(y.iloc[valid_index], prediction),
                'convergence_warnings': fold_convergence_warnings,
                'elapsed_seconds': perf_counter() - started,
            }
            per_fold.append(row)
            print(
                f"[{case_name}] seed={seed} fold={fold}/{N_SPLITS} "
                f"f1={row['f1_macro']:.5f} features={feature_count:,} "
                f"warnings={fold_convergence_warnings} "
                f"time={row['elapsed_seconds']:.1f}s"
            )

        per_seed.append({
            'case': case_name,
            'seed': seed,
            'oof_f1_macro': f1_score(
                y, oof_prediction, average='macro', zero_division=0,
            ),
            'oof_accuracy': accuracy_score(y, oof_prediction),
        })

    seed_frame = pd.DataFrame(per_seed)
    fold_frame = pd.DataFrame(per_fold)
    summary = {
        'case': case_name,
        'seeds': list(seeds),
        'oof_f1_macro_mean': seed_frame['oof_f1_macro'].mean(),
        'oof_f1_macro_std': seed_frame['oof_f1_macro'].std(ddof=0) if len(seed_frame) > 1 else np.nan,
        'oof_accuracy_mean': seed_frame['oof_accuracy'].mean(),
        'fold_f1_macro_mean': fold_frame['f1_macro'].mean(),
        'fold_f1_macro_std': fold_frame['f1_macro'].std(ddof=0),
        'feature_count_min': fold_frame['feature_count'].min(),
        'feature_count_max': fold_frame['feature_count'].max(),
        'convergence_warning_count': convergence_warning_count,
        'elapsed_seconds': fold_frame['elapsed_seconds'].sum(),
    }
    print(
        f"완료: {case_name} OOF Macro F1="
        f"{summary['oof_f1_macro_mean']:.5f}"
    )
    return {'summary': summary, 'per_seed': seed_frame, 'per_fold': fold_frame}

screen_results = {}

## 1차 비교 — case 01~05

아래 누적 후보를 모두 실행합니다. 결과표가 생성되면 leaderboard 1·2·3위를 자동으로 3-seed 확인합니다.


In [ ]:
case = 'case_01_exp009_pair_raw'
screen_results[case] = evaluate_case(case, candidates[case])


In [ ]:
case = 'case_02_plus_pair_log1p'
screen_results[case] = evaluate_case(case, candidates[case])


In [ ]:
case = 'case_03_plus_S'
screen_results[case] = evaluate_case(case, candidates[case])


In [ ]:
case = 'case_04_plus_train_contrast'
screen_results[case] = evaluate_case(case, candidates[case])


In [ ]:
case = 'case_05_plus_train_exact_top4'
screen_results[case] = evaluate_case(case, candidates[case])


## 1차 leaderboard와 자동 후보 선정

In [ ]:
leaderboard = (
    pd.DataFrame([result['summary'] for result in screen_results.values()])
    .sort_values('oof_f1_macro_mean', ascending=False)
    .reset_index(drop=True)
)
reference_case = 'case_01_exp009_pair_raw'
reference_f1 = leaderboard.loc[
    leaderboard['case'].eq(reference_case), 'oof_f1_macro_mean'
].iloc[0]
leaderboard['delta_vs_reference'] = (
    leaderboard['oof_f1_macro_mean'] - reference_f1
)
leaderboard.to_csv(RESULTS_DIR / 'leaderboard_seed42.csv', index=False)
display(leaderboard)

confirmation_cases = leaderboard.head(3)['case'].tolist()
print('3-seed confirmation cases:', confirmation_cases)


## 3-seed 확인

`confirmation_cases`를 직접 수정할 수 있습니다. 아래 셀은 선택한 case만 실행합니다.

In [ ]:
confirmation_results = {}
for case in confirmation_cases:
    confirmation_results[case] = evaluate_case(
        case,
        candidates[case],
        seeds=CONFIRMATION_SEEDS,
    )

confirmation_leaderboard = (
    pd.DataFrame([
        result['summary'] for result in confirmation_results.values()
    ])
    .sort_values('oof_f1_macro_mean', ascending=False)
    .reset_index(drop=True)
)
confirmation_leaderboard.to_csv(
    RESULTS_DIR / 'leaderboard_confirmation.csv', index=False
)
display(confirmation_leaderboard)